In [1]:
# Parameters
base_path = "C:\\Users\\ander\\OneDrive\\TCC_USP"
run_id = "20251118_143535"


In [2]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "11_event_study_latency"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


BASE_PATH: C:\Users\ander\OneDrive\TCC_USP
PROC_PATH: C:\Users\ander\OneDrive\TCC_USP\data_processed


In [3]:
# 2) Carregar dados (IBOV + notícias limpas) e normalizar
import pandas as pd, numpy as np

ibov_file = os.path.join(PROC_PATH, "ibovespa_clean.csv")
news_file = os.path.join(PROC_PATH, "noticias_real_clean.csv")

df_ibov = pd.read_csv(ibov_file)
df_news = pd.read_csv(news_file)

# padroniza colunas e datas
for df in [df_ibov, df_news]:
    if "date" in df.columns:
        df.rename(columns={"date":"data"}, inplace=True)
    df["data"] = pd.to_datetime(df["data"], errors="coerce")

df_ibov = df_ibov.sort_values("data").dropna(subset=["data","close"]).copy()
df_news = df_news.dropna(subset=["data"]).copy()
if "fonte" not in df_news.columns:
    df_news["fonte"] = "desconhecida"

print("IBOV:", df_ibov.shape, "| NEWS:", df_news.shape)
display(df_ibov.head(2)); display(df_news.head(2))

IBOV: (20, 2) | NEWS: (428, 6)


,data,close
0,2024-01-02,130749.73
1,2024-01-03,130673.55


,data,fonte,titulo,texto,texto_completo,clean_text
0,2025-09-27,IGN,"""Apenas 1 em cada 10 mil jogadores encontraria...",Esqueceu ou decidiu não comprar a Pena? O game...,"""Apenas 1 em cada 10 mil jogadores encontraria...",apenas cada mil jogadores encontraria jogou si...
1,2025-09-27,Sapo.pt,Google Fotos vai ao Tinder copiar uma nova fun...,O Google Fotos poderá receber uma funcionalida...,Google Fotos vai ao Tinder copiar uma nova fun...,google fotos vai tinder copiar nova funcionali...


In [4]:
# 3) Preparar retornos do IBOV e alinhar intervalo de datas
df_ibov["ret"] = df_ibov["close"].pct_change()
df_ibov = df_ibov.dropna(subset=["ret"]).reset_index(drop=True)

# janela do estudo (D0+1 … D0+HORIZON)
HORIZON = 5

# FILTRO CRÍTICO: só mantenha notícias dentro do intervalo do IBOV
ibov_min, ibov_max = df_ibov["data"].min(), df_ibov["data"].max()
df_news = df_news[(df_news["data"] >= ibov_min) & (df_news["data"] <= ibov_max)].copy()

print(f"Janela IBOV: {ibov_min} → {ibov_max}")
print("Notícias após filtro por janela do IBOV:", df_news.shape)

Janela IBOV: 2024-01-03 00:00:00 → 2024-01-29 00:00:00
Notícias após filtro por janela do IBOV: (0, 6)


In [5]:
# 4) Mapear cada notícia ao próximo dia útil (pregão)
mkt_days_unique = df_ibov["data"].dt.normalize().unique()
mkt_set = set(pd.to_datetime(mkt_days_unique))
last_trading_day = pd.to_datetime(df_ibov["data"].max()).normalize()

def next_trading_day(d, max_lookahead=60):
    d0 = pd.to_datetime(d)
    if pd.isna(d0):
        return pd.NaT
    d0 = d0.normalize()
    # evita procurar além do último pregão conhecido
    for _ in range(max_lookahead):
        if d0 in mkt_set:
            return d0
        d0 += pd.Timedelta(days=1)
        if d0 > last_trading_day:
            return pd.NaT
    return pd.NaT

df_news["event_day"] = df_news["data"].apply(next_trading_day)
df_news = df_news.dropna(subset=["event_day"]).sort_values("event_day")
print("Notícias válidas mapeadas para pregões:", df_news.shape)
display(df_news.head())

Notícias válidas mapeadas para pregões: (0, 7)


,data,fonte,titulo,texto,texto_completo,clean_text,event_day


In [6]:
# 5) Construir eventos (CAR e T½) de forma segura
def compute_event_car(event_date):
    # acha o índice do dia do evento no IBOV
    idx = df_ibov.index[df_ibov["data"].dt.normalize() == event_date]
    if len(idx)==0:
        return None
    i0 = idx[0]
    fut = df_ibov.iloc[i0+1 : i0+1+HORIZON]  # D+1...D+H
    if fut.empty:
        return None
    car = fut["ret"].cumsum().values  # CAR(1..H)
    car_abs = np.abs(car)
    if car_abs.size == 0:
        return None
    car_max = car_abs.max()
    if car_max == 0:
        t_half = np.nan
    else:
        t_half = next((k for k, v in enumerate(car_abs, start=1) if v >= 0.5*car_max), np.nan)
    return car, car_max, t_half

records = []
for _, row in df_news.iterrows():
    out = compute_event_car(row["event_day"])
    if out is None:
        continue
    car, car_max, t_half = out
    records.append({
        "data_noticia": row["data"],
        "event_day": row["event_day"],
        "fonte": row.get("fonte","desconhecida"),
        "titulo": row.get("titulo", row.get("title","")),
        "car_series": car,
        "car_max_abs": float(car_max),
        "t_half": float(t_half) if not pd.isna(t_half) else np.nan
    })

cols = ["data_noticia","event_day","fonte","titulo","car_series","car_max_abs","t_half"]
df_events = pd.DataFrame(records, columns=cols)
print("Eventos válidos:", df_events.shape)
display(df_events.head())

Eventos válidos: (0, 7)


,data_noticia,event_day,fonte,titulo,car_series,car_max_abs,t_half


In [7]:
# 6) Agregações (com guardas)
def daypart(ts):
    try:
        h = pd.to_datetime(ts).hour
    except Exception:
        return "desconhecido"
    if 6  <= h < 12: return "manhã"
    if 12 <= h < 18: return "tarde"
    if 18 <= h < 24: return "noite"
    return "madrugada"

df_news["daypart"] = df_news["data"].apply(daypart)

if not df_events.empty:
    # junta faixa de horário à tabela de eventos
    join_rhs = (
        df_news[["data","event_day","fonte","daypart"]]
        .drop_duplicates(["data","event_day","fonte"])
        .rename(columns={"data":"data_noticia"})
    )
    df_events = df_events.merge(
        join_rhs,
        on=["data_noticia","event_day","fonte"],
        how="left"
    )
else:
    df_events["daypart"] = pd.Series(dtype=object)

if df_events.empty:
    agg_fonte  = pd.DataFrame(columns=["fonte","n_eventos","t_half_mediana","t_half_media"])
    agg_daypart= pd.DataFrame(columns=["daypart","n_eventos","t_half_mediana","t_half_media"])
else:
    agg_fonte = (
        df_events.groupby("fonte", as_index=False)["t_half"]
        .agg(["count","median","mean"]).reset_index()
        .rename(columns={"count":"n_eventos","median":"t_half_mediana","mean":"t_half_media"})
    )
    agg_fonte.columns = ["fonte","n_eventos","t_half_mediana","t_half_media"]

    agg_daypart = (
        df_events.groupby("daypart", as_index=False)["t_half"]
        .agg(["count","median","mean"]).reset_index()
        .rename(columns={"count":"n_eventos","median":"t_half_mediana","mean":"t_half_media"})
    )
    agg_daypart.columns = ["daypart","n_eventos","t_half_mediana","t_half_media"]

print("Resumo por fonte:")
display(agg_fonte.sort_values("t_half_mediana") if not agg_fonte.empty else agg_fonte)
print("Resumo por faixa de horário:")
display(agg_daypart.sort_values("t_half_mediana") if not agg_daypart.empty else agg_daypart)

Resumo por fonte:

,fonte,n_eventos,t_half_mediana,t_half_media


Resumo por faixa de horário:


,daypart,n_eventos,t_half_mediana,t_half_media


In [8]:
# 7) CAR médio (t=1..H) geral e por fonte (com guardas)
if df_events.empty:
    car_mean_all = np.full(HORIZON, np.nan)
    car_by_source = {}
else:
    car_mat = []
    meta = []
    for _, ev in df_events.iterrows():
        c = np.asarray(ev["car_series"], dtype=float)
        if c.size < HORIZON:
            c = np.pad(c, (0, HORIZON-c.size), constant_values=np.nan)
        car_mat.append(c)
        meta.append(ev["fonte"])
    car_mat = np.array(car_mat)  # n_events x HORIZON
    car_mean_all = np.nanmean(car_mat, axis=0)

    counts = pd.Series(meta).value_counts()
    top_sources = counts.head(5).index.tolist()
    car_by_source = {}
    for s in top_sources:
        idx = [i for i,m in enumerate(meta) if m==s]
        car_by_source[s] = np.nanmean(car_mat[idx,:], axis=0)

print("CAR médio (t=1..H):", car_mean_all)
for s,c in car_by_source.items():
    print(f"Fonte={s} | CAR médio:", c)

CAR médio (t=1..H): [nan nan nan nan nan]


In [9]:
# 8) Exportar resultados para o dashboard (com fallback)
out_events = os.path.join(PROC_PATH, "latency_events.parquet")
out_fonte  = os.path.join(PROC_PATH, "latency_by_source.csv")
out_dayprt = os.path.join(PROC_PATH, "latency_by_daypart.csv")
out_carm   = os.path.join(PROC_PATH, "car_mean_all.npy")
out_cars   = os.path.join(PROC_PATH, "car_mean_by_source.npy")

# parquet → csv fallback
try:
    import pyarrow  # noqa
    df_events.to_parquet(out_events, index=False)
except Exception:
    out_events = os.path.join(PROC_PATH, "latency_events.csv")
    df_events.to_csv(out_events, index=False, encoding="utf-8-sig")

agg_fonte.to_csv(out_fonte, index=False, encoding="utf-8-sig")
agg_daypart.to_csv(out_dayprt, index=False, encoding="utf-8-sig")
np.save(out_carm, car_mean_all, allow_pickle=True)
np.save(out_cars, car_by_source, allow_pickle=True)

print("✅ Arquivos salvos para o dashboard:")
print(out_events); print(out_fonte); print(out_dayprt); print(out_carm); print(out_cars)

✅ Arquivos salvos para o dashboard:
C:\Users\ander\OneDrive\TCC_USP\data_processed\latency_events.parquet
C:\Users\ander\OneDrive\TCC_USP\data_processed\latency_by_source.csv
C:\Users\ander\OneDrive\TCC_USP\data_processed\latency_by_daypart.csv
C:\Users\ander\OneDrive\TCC_USP\data_processed\car_mean_all.npy
C:\Users\ander\OneDrive\TCC_USP\data_processed\car_mean_by_source.npy
